In [6]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages
from coffea.util import load

# ==========================================
# 1. GLOBAL CONFIGURATION & TOGGLES
# ==========================================
hep.style.use(hep.style.CMS)

COFFEA_DIR = "DataVsMCNew/" # <-- UPDATE THIS PATH
LUMI = 109.950 # target lumi in fb^-1 

# --- TOGGLES ---
COMPUTE_SYSTEMATICS = True 
PLOT_INDIVIDUAL_SYSTS = False # <-- NEW: Set to True to break down the uncertainty bands in the ratio plot

# --- K-FACTORS ---
TTBB_KFACTOR = 1.318

# ==========================================
# 2. DICTIONARIES & GROUPINGS
# ==========================================
# All Cross Sections in pb
TARGET_XSECS = {
    'tttolnu2q': 405.46, 'ttto4q': 419.31, 'ttto2l2nu': 97.90,
    'ttbbtolnu2q': 17.37, 'ttbbto4q': 19.15, 'ttbbto2l2nu': 3.75,
    'tth-hto2b': 0.331, 'tth-htonon2b': 0.238, 'ttz-ztoqq-1jets': 0.786,
    'ttwjets': 0.468, 'ttll_bin-mll-4to50': 0.039, 'ttll_bin-mll-50': 0.086, 'ttnunu': 0.164,
    'tttt': 0.0097, 'thw': 0.017, 'thq': 0.084, 'tzq_ll': 0.080,
    'st_tw_top-4q': 19.95, 'st_tw_antitop-4q': 19.95, 'st_tw_top-lnu2q': 19.29, 'st_tw_antitop-lnu2q': 19.29,
    'st_tw_top-2l2nu': 4.66, 'st_tw_antitop-2l2nu': 4.66, 'st_top_s_lep': 2.278, 'st_antitop_s_lep': 1.43,
    'st_top_t_2q': 97.74, 'st_antitop_t_2q': 58.78, 'st_top_t_lnu': 47.26, 'st_antitop_t_lnu': 28.42,
    'wtolnu-4jets_bin-1j': 9141.0, 'wtolnu-4jets_bin-2j': 2931.0, 'wtolnu-4jets_bin-3j': 864.6, 'wtolnu-4jets_bin-4j': 417.8,
    'dyto2e_mll-10to50': 5743.0, 'dyto2e_mll-50': 1827.0, 'dyto2mu_mll-10to50': 5803.0, 'dyto2mu_mll-50': 1815.0,
    'dyto2tau_mll-10to50': 5806.0, 'dyto2tau_mll-50': 1816.0,
    'ww': 125.8, 'wz': 54.7, 'zz': 16.7,
    'qcd-ht_40_70': 312300000.0, 'qcd-ht_70_100': 58470000.0, 'qcd-ht_100_200': 25310000.0,
    'qcd-ht_200_400': 1960000.0, 'qcd-ht_400_600': 97400.0, 'qcd-ht_600_800': 13560.0,
    'qcd-ht_800_1000': 3010.0, 'qcd-ht_1000_1200': 890.3, 'qcd-ht_1200_1500': 384.8,
    'qcd-ht_1500_2000': 127.3, 'qcd-ht_2000': 26.26
}

# Unskimmed Genweights to override missing data
TARGET_GENWEIGHTS = {
    'tttolnu2q': 480547550.0, 'ttto2l2nu': 466318140.0, 'ttto4q': 468705400.0,
    'ttbbtolnu2q': 21007254.0, 'ttbbto2l2nu': 11683236.0, 'ttbbto4q': 14012432.0
}

# Process Groupings
bkg_processes = ['VJets', 'QCD', 'tt_B', 'TTBar', 'SingleTop', 'TTX'] 
sig_processes = ['ttZ', 'ttH']
all_processes = bkg_processes + sig_processes + ['data_obs']

process_labels = {
    'VJets': 'V+Jets', 'QCD': 'QCD', 'tt_B': r'$t\bar{t}+bb$',
    'TTBar': r'$t\bar{t} + lf/cc$', 'SingleTop': 'Single Top',
    'TTX': r'$t\bar{t}X$', 'ttZ': r'$t\bar{t}Z$', 'ttH': r'$t\bar{t}H$', 
    'data_obs': 'Data'
}

bkg_colors = {'VJets':'#3f90da', 'QCD':'#ffa90e', 'tt_B':'#bd1f01', 'TTBar':'#94a4a2', 'SingleTop':'#e76300', 'TTX':'#b9ac70'}
sig_colors = {'ttZ':'#832db6', 'ttH':'#a96b59'}
mc_colors = {**bkg_colors, **sig_colors}

# Variables
validation_vars = [
    #'nPV', 
    'nPVGood', 'MET_pt', 'MET_phi', 'lep_pt', 'lep_eta',
    'ele_pt', 'ele_eta', 'muon_pt', 'muon_eta',
    'n_ak4jets', 'n_bjet', 'n_ak8',
    'jet1_pt', 'jet1_eta', 'jet2_pt', 'jet2_eta',
    'bjet1_pt', #'bjet2_pt', 
    'bjet1_eta', #'bjet2_eta', 
    'jet1_btag', 'jet2_btag', 'bjet1_btag', #'bjet2_btag',
    'fatjet1_pt', 'fatjet1_eta', 'fatjet1_mass', 'ZH_bbvLscore'
]
cut_vars = ['ZH_bbvLscore', 'ZH_M', 'ZH_pt', 'n_b_outZH', 'n_ak4jets', 'MET_pt', 'muon_eta']

weight_vars = [
    'genWeight', 'topptWeight', 'topptWeight_Up', 'topptWeight_Down',
    'ele_reco_sf', 'ele_reco_sfup', 'ele_reco_sfdown', 'ele_id_sf', 'ele_id_sfup', 'ele_id_sfdown',
    'ele_trig_sf', 'ele_trig_sfup', 'ele_trig_sfdown', 'mu_id_sf', 'mu_id_sfup', 'mu_id_sfdown',
    'mu_iso_sf', 'mu_iso_sfup', 'mu_iso_sfdown', 'mu_trig_sf', 'mu_trig_sfup', 'mu_trig_sfdown',
    'puWeight', 'puWeight_up', 'puWeight_down', 'isr_up', 'isr_down', 'fsr_up', 'fsr_down',
    'mu_r_up', 'mu_r_down', 'mu_f_up', 'mu_f_down', 'mu_rf_up', 'mu_rf_down',
    'btag_sf', 'btag_sfup', 'btag_sfdown'
]

weight_syst_mapping = {
    'topptWeight': ('topptWeight', 'topptWeight_Up', 'topptWeight_Down'),
    'btag_sf': ('btag_sf', 'btag_sfup', 'btag_sfdown'),
    'ele_reco_sf': ('ele_reco_sf', 'ele_reco_sfup', 'ele_reco_sfdown'),
    'ele_id_sf':   ('ele_id_sf', 'ele_id_sfup', 'ele_id_sfdown'),
    'ele_trig_sf': ('ele_trig_sf', 'ele_trig_sfup', 'ele_trig_sfdown'),
    'mu_id_sf':    ('mu_id_sf', 'mu_id_sfup', 'mu_id_sfdown'),
    'mu_iso_sf':   ('mu_iso_sf', 'mu_iso_sfup', 'mu_iso_sfdown'),
    'mu_trig_sf':  ('mu_trig_sf', 'mu_trig_sfup', 'mu_trig_sfdown'),
    'puWeight':    ('puWeight', 'puWeight_up', 'puWeight_down'),
    'isr':         (None, 'isr_up', 'isr_down'), 
    'fsr':         (None, 'fsr_up', 'fsr_down'),
    'mu_r':        (None, 'mu_r_up', 'mu_r_down'),
    'mu_f':        (None, 'mu_f_up', 'mu_f_down'),
    'mu_rf':       (None, 'mu_rf_up', 'mu_rf_down')
}

syst_bases = [
    'AK4PFPuppi_JER', 'AK8PFPuppi_JER', 'AK4PFPuppi_JES_Total', 'AK8PFPuppi_JES_Total',  
    # 'AK4PFPuppi_JES_AbsoluteMPFBias', 'AK4PFPuppi_JES_AbsoluteScale',
    # 'AK4PFPuppi_JES_AbsoluteStat', 'AK4PFPuppi_JES_FlavorQCD', 'AK4PFPuppi_JES_Fragmentation',
    # 'AK4PFPuppi_JES_PileUpDataMC', 'AK4PFPuppi_JES_PileUpEnvelope', 'AK4PFPuppi_JES_PileUpMuZero',
    # 'AK4PFPuppi_JES_PileUpPtBB', 'AK4PFPuppi_JES_PileUpPtEC1', 'AK4PFPuppi_JES_PileUpPtEC2',
    # 'AK4PFPuppi_JES_PileUpPtHF', 'AK4PFPuppi_JES_PileUpPtRef', 'AK4PFPuppi_JES_RelativeBal',
    # 'AK4PFPuppi_JES_RelativeFSR', 'AK4PFPuppi_JES_RelativeJEREC1', 'AK4PFPuppi_JES_RelativeJEREC2',
    # 'AK4PFPuppi_JES_RelativeJERHF', 'AK4PFPuppi_JES_RelativePtBB', 'AK4PFPuppi_JES_RelativePtEC1',
    # 'AK4PFPuppi_JES_RelativePtEC2', 'AK4PFPuppi_JES_RelativePtHF', 'AK4PFPuppi_JES_RelativeSample',
    # 'AK4PFPuppi_JES_RelativeStatEC', 'AK4PFPuppi_JES_RelativeStatFSR', 'AK4PFPuppi_JES_RelativeStatHF',
    # 'AK4PFPuppi_JES_SinglePionECAL', 'AK4PFPuppi_JES_SinglePionHCAL', 'AK4PFPuppi_JES_TimePtEta',
    # 'AK8PFPuppi_JER', 'AK8PFPuppi_JES_AbsoluteMPFBias', 'AK8PFPuppi_JES_AbsoluteScale',
    # 'AK8PFPuppi_JES_AbsoluteStat', 'AK8PFPuppi_JES_FlavorQCD', 'AK8PFPuppi_JES_Fragmentation',
    # 'AK8PFPuppi_JES_PileUpDataMC', 'AK8PFPuppi_JES_PileUpEnvelope', 'AK8PFPuppi_JES_PileUpMuZero',
    # 'AK8PFPuppi_JES_PileUpPtBB', 'AK8PFPuppi_JES_PileUpPtEC1', 'AK8PFPuppi_JES_PileUpPtEC2',
    # 'AK8PFPuppi_JES_PileUpPtHF', 'AK8PFPuppi_JES_PileUpPtRef', 'AK8PFPuppi_JES_RelativeBal',
    # 'AK8PFPuppi_JES_RelativeFSR', 'AK8PFPuppi_JES_RelativeJEREC1', 'AK8PFPuppi_JES_RelativeJEREC2',
    # 'AK8PFPuppi_JES_RelativeJERHF', 'AK8PFPuppi_JES_RelativePtBB', 'AK8PFPuppi_JES_RelativePtEC1',
    # 'AK8PFPuppi_JES_RelativePtEC2', 'AK8PFPuppi_JES_RelativePtHF', 'AK8PFPuppi_JES_RelativeSample',
    # 'AK8PFPuppi_JES_RelativeStatEC', 'AK8PFPuppi_JES_RelativeStatFSR', 'AK8PFPuppi_JES_RelativeStatHF',
    # 'AK8PFPuppi_JES_SinglePionECAL', 'AK8PFPuppi_JES_SinglePionHCAL', 'AK8PFPuppi_JES_TimePtEta',
    'ele_scale', 'ele_smear', 'unclust_En', 'muon_scale', 'muon_smear'
]
systematics = ['nominal'] + ([f"{s}Up" for s in syst_bases] + [f"{s}Down" for s in syst_bases] if COMPUTE_SYSTEMATICS else [])
all_syst_bases = syst_bases + list(weight_syst_mapping.keys()) if COMPUTE_SYSTEMATICS else []

binning_dict = {
    'nPV': (50, 0, 100), 'nPVGood': (50, 0, 100),
    'MET_pt': (40, 0, 800), 'MET_phi': (30, -3.14, 3.14),
    'lep_pt': (40, 0, 800), 'ele_pt': (40, 0, 800), 'muon_pt': (40, 0, 800),
    'lep_eta': (30, -2.5, 2.5), 'ele_eta': (30, -2.5, 2.5), 'muon_eta': (30, -2.4, 2.4),
    'n_ak4jets': (15, 0, 15), 'n_bjet': (10, 0, 10), 'n_ak8': (5, 0, 5),
    'jet1_pt': (40, 0, 1000), 'jet2_pt': (40, 0, 800),
    'bjet1_pt': (40, 0, 800), 'bjet2_pt': (40, 0, 600),
    'jet1_eta': (30, -2.5, 2.5), 'jet2_eta': (30, -2.5, 2.5),
    'bjet1_eta': (30, -2.5, 2.5), 'bjet2_eta': (30, -2.5, 2.5),
    'jet1_btag': (20, 0, 1), 'jet2_btag': (20, 0, 1),
    'bjet1_btag': (20, 0, 1), 'bjet2_btag': (20, 0, 1),
    'fatjet1_pt': (40, 200, 1200), 'fatjet1_eta': (30, -2.5, 2.5), 'fatjet1_mass': (40, 0, 400),
    'ZH_bbvLscore': (30, 0, 1)
}
default_binning = (40, 0, 500)

# ==========================================
# 3. KINEMATIC CUTS & WEIGHT CALCULATOR
# ==========================================
def cuts(df_):
    return (
        (df_['n_ak4jets']  >= 5)       &
        (df_['n_b_outZH']  >= 2)       &
        (df_['ZH_pt']      >= 200)     &
        (df_['ZH_M']       >= 50)      &
        (df_['MET_pt']     > 20)       &
        #(df_['ZH_bbvLscore'] > 0.5) &
        (df_['fatjet1_mass'] < 200) &
        #(df_['muon_eta'] > 0) &
        (df_['ZH_M']       <= 200)
    )

def getZhbbWeight(df_):
    """Calculates Final Weight. DF already contains pedantic dataset_norm_weight."""
    if 'dataset_norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index) 
        
    weight = df_['dataset_norm_weight'].copy()
    
    gen_w = df_.get('genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)
    
    sfs = ['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'mu_trig_sf', 'puWeight','ele_trig_sf', 'topptWeight']#, 'btag_sf']
    for sf in sfs:
        if sf in df_.columns:
            # --- THE ARMOR ---
            # If an efficiency map divides by near-zero, clip the resulting massive SF to 10.0
            # If an SF drops to exactly zero, floor it at 0.01 to prevent killing the event entirely
            safe_sf = np.clip(df_[sf].fillna(1.0), 0.01, 10.0)
            weight *= safe_sf
        
    return weight

# ==========================================
# 4. DATA LOADER
# ==========================================

print("Building dynamic genweight cache from nominal files...")
NOMINAL_GW_CACHE = {}
for f in glob.glob(os.path.join(COFFEA_DIR, "*nom*.coffea")):
    try:
        filein = load(f)
        gw_dict = filein.get('sum_signOf_genweights', {})
        for dataset, gw in gw_dict.items():
            # Handle both nested dictionaries and flat values
            NOMINAL_GW_CACHE[dataset] = gw.get(dataset, 1.0) if isinstance(gw, dict) else gw
    except Exception:
        continue
print(f"Cached true genweights for {len(NOMINAL_GW_CACHE)} datasets.")

def get_mapped_proc(raw_proc):
    p = raw_proc.lower()
    if 'data' in p: return 'data_obs'
    if 'ttto' in p:
        if '__' not in p: return None  # VETO inclusive parent!
        if 'tt+b' in p: return None    # VETO 5FS heavy flavor
        return 'TTBar'                 # KEEP tt+LF and tt+C
        
    if 'ttbb' in p: 
        if '__' not in p: return None  # VETO inclusive parent!
        if 'tt+b' in p: return 'tt_B'  # KEEP 4FS heavy flavor
        return None                    # VETO 4FS light flavor tails
        
    # --- VJets and Others ---
    if 'wjets' in p or 'dyjets' in p: 
        # if '__' not in p: return None
        return 'VJets'
    if 'qcd' in p: return 'QCD'
    if 'tth' in p: return 'ttH'
    if 'ttz' in p or 'ttll' in p or 'ttnunu' in p: return 'ttZ'
    if 'singletop' in p: return 'SingleTop'
    if 'ttx' in p: return 'TTX'
    if 'vv' in p: return 'VV'
    return None

def load_and_cut_data(variation='nominal', coffea_dir=COFFEA_DIR):
    tracked_data = {proc: [] for proc in all_processes}
    all_files = glob.glob(os.path.join(coffea_dir, "*.coffea"))
    
    file_mapping = {
        'ele_scale': 'shape_electron_scale_and_smearing',
        'ele_smear': 'shape_electron_scale_and_smearing',
        'muon_scale': 'shape_muons_scale_and_resolution',
        'muon_smear': 'shape_muons_scale_and_resolution',
        'unclust_En': 'shape_met_type1_calibration'
    }
    
    if variation == 'nominal':
        valid_files = [f for f in all_files if 'nom' in os.path.basename(f).lower()]
    else:
        # Strip Up/Down suffix to find the base systematic name
        if variation.endswith('Up'):
            base_syst = variation[:-2]
        elif variation.endswith('Down'):
            base_syst = variation[:-4]
        else:
            base_syst = variation
        
        # Handle JEC/JER replacements or fallback to the file mapping dictionary
        if 'AK4PFPuppi_' in base_syst or 'AK8PFPuppi_' in base_syst:
            search_string = base_syst.replace('AK4PFPuppi_', 'jec_').replace('AK8PFPuppi_', 'jec_')
        else:
            search_string = file_mapping.get(base_syst, base_syst)
            
        valid_files = [f for f in all_files if search_string.lower() in os.path.basename(f).lower()]
        
        if len(valid_files) == 0:
            print(f"  -> [WARNING] Searched for '{search_string}' but found nothing!")

    print(f"Extracting '{variation}' from {len(valid_files)} files...")

    for file_path in valid_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        
        for raw_proc in filein['columns'].keys():
            mapped_proc = get_mapped_proc(raw_proc)
            if mapped_proc not in all_processes: continue
                
            for dataset in filein['columns'][raw_proc].keys():
                clean_name = dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                
            # --- PEDANTIC NORMALIZATION CALCULATION ---
                if mapped_proc != 'data_obs':
                    if clean_name not in TARGET_XSECS:
                        print(f"🚨 Missing XSEC for {clean_name}! Event weights will be zeroed out.")
                        xsec = 0
                    else:
                        xsec = TARGET_XSECS[clean_name]
                        
                    # 1. Check forced overrides (TTBar/tt_B)
                    sum_gw = TARGET_GENWEIGHTS.get(clean_name)
                    
                    if not sum_gw:
                        # 2. Check the current file's dictionary
                        file_gw = genweight_dict.get(dataset)
                        sum_gw = file_gw.get(dataset) if isinstance(file_gw, dict) else file_gw
                        
                        # 3. THE RESCUE: If the systematic file dropped it, pull from the nominal cache
                        if not sum_gw or sum_gw == 1.0:
                            sum_gw = NOMINAL_GW_CACHE.get(dataset, 1.0)
                            
                        if sum_gw == 1.0:
                            pass # Suppressed warning for cleanliness
                            
                    dataset_norm_weight = (xsec * LUMI * 1000) / sum_gw
                else:
                    dataset_norm_weight = 1.0
                # ------------------------------------------

                try:
                    base_dict = filein['columns'][raw_proc][dataset]['btag_mask'][variation]
                    nom_dict = filein['columns'][raw_proc][dataset]['btag_mask']['nominal']
                except KeyError:
                    continue 

                tmp_data = {}
                extract_list = validation_vars + cut_vars + (weight_vars if mapped_proc != 'data_obs' else [])
                
                for var in extract_list:
                    dict_key = f'spanet_output_{var}' if var in ['ttzbb', 'tthbb', 'ttbb', 'ttlf', 'ttcc', 'signal'] else f'events_{var}'
                    
                    if dict_key in base_dict:
                        tmp_data[var] = np.array(base_dict[dict_key].value)
                    elif dict_key in nom_dict:
                        tmp_data[var] = np.array(nom_dict[dict_key].value)

                if tmp_data: 
                    df = pd.DataFrame(tmp_data)
                    df['dataset_norm_weight'] = dataset_norm_weight
                    tracked_data[mapped_proc].append(df)

    data_dict = {}
    for proc in all_processes:
        if tracked_data[proc]:
            df = pd.concat(tracked_data[proc], ignore_index=True)
            df = df[cuts(df)] 
            df['tot_weight'] = getZhbbWeight(df) if proc != 'data_obs' else 1.0
            # --- APPLY K-FACTORS ---
            if proc == 'tt_B':
                df['tot_weight'] *= TTBB_KFACTOR
            # -----------------------
            data_dict[proc] = df
        else:
            data_dict[proc] = None
            
    return data_dict

# ==========================================
# 5. HISTOGRAM ACCUMULATION
# ==========================================
print("\n--- Starting Data Extraction & Histogramming ---")
hist_dict = {v: {p: {} for p in all_processes} for v in validation_vars}

for syst in systematics:
    current_data = load_and_cut_data(variation=syst)
    
    if syst == 'nominal':
        print("\n RAW UNWEIGHTED COUNTS (PASSING CUTS)")
        for proc in all_processes:
            df = current_data[proc]
            print(f" {proc:<15} : {len(df) if df is not None else 0:,} events")
        print("="*40 + "\n")
    
    for var in validation_vars:
        n_bins, x_min, x_max = binning_dict.get(var, default_binning)
        bins = np.linspace(x_min, x_max, n_bins + 1)
        
        for proc in all_processes:
            df = current_data[proc]
            if df is None or df.empty or var not in df.columns:
                # ONLY force zeros if it's the nominal distribution
                if syst == 'nominal':
                    hist_dict[var][proc][syst] = {'counts': np.zeros(n_bins), 'err2': np.zeros(n_bins)}
                continue
                
            mask = df[var].notna()
            vals = np.clip(df[var][mask], None, bins[-1])
            weights = df['tot_weight'][mask].copy()
            
            # --- Nominal Shapes ---
            counts, _ = np.histogram(vals, bins=bins, weights=weights)
            err2, _ = np.histogram(vals, bins=bins, weights=weights**2)
            hist_dict[var][proc][syst] = {'counts': counts, 'err2': err2}
            
            # --- Compute Weight Systematics ---
            if syst == 'nominal' and proc != 'data_obs' and COMPUTE_SYSTEMATICS:
                for w_syst, (nom_col, up_col, dn_col) in weight_syst_mapping.items():
                    
                    # 1. Create a safe fallback Series of 1.0s
                    default_ones = pd.Series(1.0, index=df.index)
                    
                    # 2. Extract the columns safely.
                    nom_s = df.get(nom_col, default_ones) if nom_col else default_ones
                    up_s  = df.get(up_col, nom_s) if up_col else nom_s
                    dn_s  = df.get(dn_col, nom_s) if dn_col else nom_s
                    
                    # 3. Prevent division by zero in the nominal weights
                    safe_nom = np.where(nom_s[mask] == 0, 1.0, nom_s[mask]) 
                    
                    # 4. Calculate the shifted weights with clipping to prevent explosive physics outliers
                    up_w = weights * np.clip(up_s[mask] / safe_nom, 0.1, 10.0)
                    dn_w = weights * np.clip(dn_s[mask] / safe_nom, 0.1, 10.0)
                    
                    hist_dict[var][proc][f'{w_syst}Up'] = {
                        'counts': np.histogram(vals, bins=bins, weights=up_w)[0], 
                        'err2': np.histogram(vals, bins=bins, weights=up_w**2)[0]
                    }
                    hist_dict[var][proc][f'{w_syst}Down'] = {
                        'counts': np.histogram(vals, bins=bins, weights=dn_w)[0], 
                        'err2': np.histogram(vals, bins=bins, weights=dn_w**2)[0]
                    }
    del current_data

# ==========================================
# 6. PLOTTING FUNCTION
# ==========================================
def plot_variables_to_pdf(var_names, hist_dictionary, syst_base_list, output_filename="Data_MC_Systematics.pdf"):
    mc_processes = bkg_processes + sig_processes
    
    with PdfPages(output_filename) as pdf:
        for chunk_start in range(0, len(var_names), 8):
            chunk_vars = var_names[chunk_start : chunk_start + 8]
            
            fig = plt.figure(figsize=(32, 16))
            outer_grid = fig.add_gridspec(2, 4, wspace=0.3, hspace=0.3)
            
            for idx, var_name in enumerate(chunk_vars):
                row, col = idx // 4, idx % 4
                inner_grid = outer_grid[row, col].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.00)
                ax = fig.add_subplot(inner_grid[0])
                rax = fig.add_subplot(inner_grid[1], sharex=ax)
                ax.tick_params(labelbottom=False)
                
                n_bins, x_min, x_max = binning_dict.get(var_name, default_binning)
                bins = np.linspace(x_min, x_max, n_bins + 1)
                bin_centers = 0.5 * (bins[1:] + bins[:-1])
                
                mc_hists, mc_labels, mc_colors_list = [], [], []
                total_mc_counts = np.zeros(n_bins)
                total_mc_stat_err2 = np.zeros(n_bins)
                total_mc_syst_err2 = np.zeros(n_bins)
                
                # Dictionary to store relative fractional error of individual systematics for plotting
                individual_rel_errs = {} 
                
                # Stack Nominal MC
                for proc in mc_processes:
                    nom_data = hist_dictionary[var_name][proc].get('nominal')
                    if not nom_data: continue
                        
                    counts, err2 = nom_data['counts'], nom_data['err2']
                    yield_total, stat_unc = np.sum(counts), np.sqrt(np.sum(err2))
                    
                    mc_hists.append(counts)
                    mc_labels.append(f"{process_labels.get(proc, proc)} ({yield_total:.1f} ± {stat_unc:.1f})")
                    mc_colors_list.append(mc_colors[proc])
                    
                    total_mc_counts += counts
                    total_mc_stat_err2 += err2

                # Systematics Envelope
                if COMPUTE_SYSTEMATICS:
                    for s_base in syst_base_list:
                        syst_up_diff, syst_dn_diff = np.zeros(n_bins), np.zeros(n_bins)
                        for proc in mc_processes:
                            nom = hist_dictionary[var_name][proc].get('nominal', {'counts': np.zeros(n_bins)})['counts']
                            
                            # Safely fetch the systematic dict, defaulting to None if missing
                            up_dict = hist_dictionary[var_name][proc].get(f'{s_base}Up')
                            dn_dict = hist_dictionary[var_name][proc].get(f'{s_base}Down')
                            
                            # Extract counts or default to nominal
                            up = up_dict['counts'] if up_dict is not None else nom
                            dn = dn_dict['counts'] if dn_dict is not None else nom

                            # --- MISSING SYSTEMATIC SAFETY NET ---
                            if np.sum(up) == 0 and np.sum(nom) > 0: up = nom
                            if np.sum(dn) == 0 and np.sum(nom) > 0: dn = nom
                            # -------------------------------------
                            
                            syst_up_diff += (up - nom)
                            syst_dn_diff += (dn - nom)
                            
                        # Capture the relative error of this specific systematic before grouping
                        safe_mc = np.where(total_mc_counts == 0, 1e-10, total_mc_counts)
                        rel_err_array = np.maximum(np.abs(syst_up_diff), np.abs(syst_dn_diff)) / safe_mc
                        individual_rel_errs[s_base] = rel_err_array
                            
                        total_mc_syst_err2 += np.maximum(np.abs(syst_up_diff), np.abs(syst_dn_diff))**2

                total_mc_err = np.sqrt(total_mc_stat_err2 + total_mc_syst_err2)

                if mc_hists:
                    hep.histplot(mc_hists, bins=bins, ax=ax, stack=True, histtype='fill', 
                                 label=mc_labels, color=mc_colors_list, sort='yield')

                ax.stairs(values=total_mc_counts + total_mc_err, 
                    baseline=np.clip(total_mc_counts - total_mc_err, 0, None),
                    edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none', 
                    label=r'Stat $\oplus$ Syst Unc.' if COMPUTE_SYSTEMATICS else 'Stat Unc.')

                # Process Data
                data_counts = hist_dictionary[var_name]['data_obs']['nominal']['counts'].astype(float)
                
                # Blinding
                if var_name == 'ZH_bbvLscore':
                    blind_mask = bin_centers > 0.8
                    data_counts[blind_mask] = np.nan
                
                data_err = np.sqrt(data_counts)
                data_yield = np.nansum(data_counts) 
                
                if data_yield > 0:
                    data_lbl = f"Data ({data_yield:.0f} ± {np.sqrt(data_yield):.1f})"
                    hep.histplot(data_counts, bins=bins, ax=ax, stack=False, histtype='errorbar', 
                                 color='black', label=data_lbl, yerr=data_err)

                # Ratio Panel
                with np.errstate(divide='ignore', invalid='ignore'):
                    ratio = data_counts / total_mc_counts
                    ratio_err = np.abs(data_err / total_mc_counts) 
                    mc_rel_err = np.abs(total_mc_err / total_mc_counts)

                for arr in [ratio, ratio_err, mc_rel_err]:
                    arr[np.isnan(arr) | np.isinf(arr)] = 0

                yerr_down = np.clip(ratio_err, 0, np.maximum(ratio, 0))

                # Plot the main total systematic band
                rax.stairs(values=1 + mc_rel_err, 
                           baseline=np.clip(1 - mc_rel_err, 0, None),
                           edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none',
                           label='Total Syst.' if PLOT_INDIVIDUAL_SYSTS else None)
                           
                # ==========================================
                # 7. BREAKDOWN PLOTTING (RATIO PANEL)
                # ==========================================
                if COMPUTE_SYSTEMATICS and PLOT_INDIVIDUAL_SYSTS:
                    # Sort systematics by their maximum percentage impact across the bins
                    sorted_systs = sorted(individual_rel_errs.items(), key=lambda item: np.max(item[1]), reverse=True)
                    
                    cmap = plt.get_cmap('tab10')
                    for i, (s_base, ind_err) in enumerate(sorted_systs):
                        ind_err[np.isnan(ind_err) | np.isinf(ind_err)] = 0
                        ind_err = np.clip(ind_err, 0, 5.0) # Cap visual vertical explosion
                        
                        # Only explicitly draw and label the top 5 driving systematics
                        if i < 5 and np.max(ind_err) > 0.005: # Must have at least a 0.5% effect
                            rax.plot(bin_centers, 1 + ind_err, color=cmap(i), linewidth=1.5, label=s_base)
                            rax.plot(bin_centers, np.clip(1 - ind_err, 0, None), color=cmap(i), linewidth=1.5)
                        # Render the remaining minor systematics as faint background ghost lines
                        elif np.max(ind_err) > 0.001:
                            rax.plot(bin_centers, 1 + ind_err, color='gray', linewidth=0.5, alpha=0.3)
                            rax.plot(bin_centers, np.clip(1 - ind_err, 0, None), color='gray', linewidth=0.5, alpha=0.3)
                            
                    # Build the secondary legend for the ratio panel
                    handles, labels = rax.get_legend_handles_labels()
                    if handles:
                        rax.legend(handles, labels, loc='upper left', fontsize=8, ncol=2)
                # ==========================================
                           
                rax.errorbar(bin_centers, ratio, yerr=[yerr_down, ratio_err], fmt='ko', markersize=3)
                rax.axhline(1, color='black', linestyle='--')
                
                # Styling
                ax.set_ylabel("Events")
                ax.legend(loc='upper right', ncol=2, fontsize=10) 
                
                max_val = max(np.max(total_mc_counts), np.max(data_counts))
                ax.set_ylim(0.1, max_val * 100 if max_val > 0 else 100)
                ax.set_yscale('log')
                
                rax.set_xlabel(var_name)
                rax.set_ylabel("Data / MC")
                rax.set_ylim(0, 2)
                
                hep.cms.label("Preliminary", data=(data_yield > 0), lumi=LUMI, ax=ax, com=13.6, fontsize=12)

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig) 
            
    print(f"\nFinished generating plots. Saved to {output_filename}")

plot_variables_to_pdf(validation_vars, hist_dict, all_syst_bases, "Data_MC_Systematics_Sources.pdf")

Building dynamic genweight cache from nominal files...
Cached true genweights for 53 datasets.

--- Starting Data Extraction & Histogramming ---
Extracting 'nominal' from 6 files...

 RAW UNWEIGHTED COUNTS (PASSING CUTS)
 VJets           : 3,812 events
 QCD             : 731 events
 tt_B            : 319,892 events
 TTBar           : 2,052,383 events
 SingleTop       : 105,306 events
 TTX             : 1,194,532 events
 ttZ             : 106,607 events
 ttH             : 573,066 events
 data_obs        : 157,156 events

Extracting 'AK4PFPuppi_JERUp' from 3 files...
Extracting 'AK8PFPuppi_JERUp' from 3 files...
Extracting 'AK4PFPuppi_JES_TotalUp' from 3 files...
Extracting 'AK8PFPuppi_JES_TotalUp' from 3 files...
Extracting 'ele_scaleUp' from 3 files...
Extracting 'ele_smearUp' from 3 files...
Extracting 'unclust_EnUp' from 3 files...
Extracting 'muon_scaleUp' from 3 files...
Extracting 'muon_smearUp' from 3 files...
Extracting 'AK4PFPuppi_JERDown' from 3 files...
Extracting 'AK8PFPuppi

In [86]:
import glob, os
from coffea.util import load

def dump_true_genweights(coffea_dir):
    nom_files = glob.glob(os.path.join(coffea_dir, "*nom*.coffea"))
    print("TARGET_GENWEIGHTS = {")
    for f in nom_files:
        filein = load(f)
        gw_dict = filein.get('sum_signOf_genweights', {})
        for raw_proc, datasets in filein['columns'].items():
            for dataset in datasets.keys():
                clean_name = dataset.lower().replace('_2024', '').replace('_2023', '').replace('_2022', '')
                file_gw = gw_dict.get(dataset, 1.0)
                sum_gw = file_gw.get(dataset, 1.0) if isinstance(file_gw, dict) else file_gw
                if sum_gw != 1.0:
                    print(f"    '{clean_name}': {sum_gw},")
    print("}")

dump_true_genweights("DataVsMCNew/")

TARGET_GENWEIGHTS = {
    'tttolnu2q': 46424012.0,
    'tttolnu2q': 46424012.0,
    'tttolnu2q': 46424012.0,
    'ttto2l2nu': 56473276.0,
    'ttto2l2nu': 56473276.0,
    'ttto2l2nu': 56473276.0,
    'ttto4q': 2610332.0,
    'ttto4q': 2610332.0,
    'ttto4q': 2610332.0,
    'ttbbtolnu2q': 2844915.0,
    'ttbbtolnu2q': 2844915.0,
    'ttbbtolnu2q': 2844915.0,
    'ttbbto2l2nu': 2123403.0,
    'ttbbto2l2nu': 2123403.0,
    'ttbbto2l2nu': 2123403.0,
    'ttbbto4q': 134167.0,
    'ttbbto4q': 134167.0,
    'ttbbto4q': 134167.0,
    'tzq_ll': 2377176.0,
    'thq': 19986024.0,
    'thw': 14995985.0,
    'tttt': 4397284.0,
    'ttwjets': 3454481.0,
    'st_antitop_t_lnu': 20789716.0,
    'st_antitop_t_2q': 40696368.0,
    'st_top_t_lnu': 41430068.0,
    'st_top_t_2q': 81985744.0,
    'st_antitop_s_lep': 4916007.0,
    'st_top_s_lep': 8000858.0,
    'st_tw_antitop-2l2nu': 14999482.0,
    'st_tw_antitop-lnu2q': 29394636.0,
    'st_tw_antitop-4q': 23999152.0,
    'st_tw_top-2l2nu': 14997466.0,
  

In [67]:
def diagnose_systematic(hist_dictionary, var_name, s_base, processes):
    print(f"\n{'='*50}")
    print(f"DIAGNOSTIC REPORT: {s_base} on variable '{var_name}'")
    print(f"{'='*50}")
    
    for proc in processes:
        nom_data = hist_dictionary[var_name][proc].get('nominal')
        if not nom_data:
            continue
            
        nom_counts = nom_data['counts']
        up_counts = hist_dictionary[var_name][proc].get(f'{s_base}Up', {'counts': nom_counts})['counts']
        dn_counts = hist_dictionary[var_name][proc].get(f'{s_base}Down', {'counts': nom_counts})['counts']
        
        nom_yield = np.sum(nom_counts)
        up_yield = np.sum(up_counts)
        dn_yield = np.sum(dn_counts)
        
        # Calculate the relative shift
        up_shift = ((up_yield - nom_yield) / nom_yield * 100) if nom_yield > 0 else 0
        dn_shift = ((dn_yield - nom_yield) / nom_yield * 100) if nom_yield > 0 else 0
        
        # Flag anything suspicious (shifts > 10% or exactly -100%)
        flag = ""
        if abs(up_shift) > 10 or abs(dn_shift) > 10:
            flag = " <-- SUSPICIOUS SHIFT"
        if up_yield == 0 or dn_yield == 0:
            flag = " <-- WARNING: VARIATION ZEROED OUT"

        print(f"\nProcess: {proc} {flag}")
        print(f"  Nominal Yield: {nom_yield:.2f}")
        print(f"  Up Yield:      {up_yield:.2f} ({up_shift:+.1f}%)")
        print(f"  Down Yield:    {dn_yield:.2f} ({dn_shift:+.1f}%)")
        
        # Check for single-bin explosions (outlier weights)
        max_diff_up = np.max(np.abs(up_counts - nom_counts))
        if max_diff_up > (nom_yield * 0.5): # If a single bin changes by 50% of the total yield
             print(f"  [!] ALERT: Massive single-bin shift detected in UP variation (Max shift: {max_diff_up:.2f})")
for i in syst_bases:
    diagnose_systematic(hist_dict, 'jet1_btag', i, bkg_processes+sig_processes)


DIAGNOSTIC REPORT: AK4PFPuppi_JER on variable 'jet1_btag'

Process: VJets  <-- WARNING: VARIATION ZEROED OUT
  Nominal Yield: 1824.68
  Up Yield:      0.00 (-100.0%)
  Down Yield:    0.00 (-100.0%)
  [!] ALERT: Massive single-bin shift detected in UP variation (Max shift: 1437.32)

Process: QCD  <-- WARNING: VARIATION ZEROED OUT
  Nominal Yield: 1277.25
  Up Yield:      0.00 (-100.0%)
  Down Yield:    0.00 (-100.0%)
  [!] ALERT: Massive single-bin shift detected in UP variation (Max shift: 824.21)

Process: tt_B 
  Nominal Yield: 28906.58
  Up Yield:      28721.18 (-0.6%)
  Down Yield:    28824.62 (-0.3%)

Process: TTBar 
  Nominal Yield: 132956.71
  Up Yield:      132240.80 (-0.5%)
  Down Yield:    132437.63 (-0.4%)

Process: SingleTop 
  Nominal Yield: 7223.09
  Up Yield:      7638.37 (+5.7%)
  Down Yield:    7577.37 (+4.9%)

Process: TTX  <-- SUSPICIOUS SHIFT
  Nominal Yield: 517.25
  Up Yield:      593.61 (+14.8%)
  Down Yield:    583.28 (+12.8%)

Process: ttZ  <-- WARNING: VARIAT

In [10]:
import scipy.stats as stats

def plot_sf_profile(data_dict, x_var, sf_var, output_filename):
    """
    Plots the average Scale Factor vs a Kinematic Variable (Profile Plot)
    """
    print(f"Creating profile plot for {sf_var} vs {x_var}...")
    
    # 1. Combine all MC processes into a single DataFrame
    mc_dfs = []
    for proc in bkg_processes + sig_processes:
        if data_dict.get(proc) is not None and not data_dict[proc].empty:
            mc_dfs.append(data_dict[proc])
            
    if not mc_dfs:
        print("No MC data found to plot!")
        return
        
    df_mc = pd.concat(mc_dfs, ignore_index=True)
    
    # 2. Drop NaNs to ensure the arrays align properly
    mask = df_mc[x_var].notna() & df_mc[sf_var].notna()
    x_vals = np.clip(df_mc[x_var][mask].values, None, binning_dict[x_var][2]) # clip to max bin
    sf_vals = df_mc[sf_var][mask].values
    
    # 3. Get binning from your existing dictionary
    n_bins, x_min, x_max = binning_dict.get(x_var, default_binning)
    bins = np.linspace(x_min, x_max, n_bins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    x_err = 0.5 * (bins[1:] - bins[:-1])
    
    # 4. Calculate Mean and Standard Error per bin
    counts, _ = np.histogram(x_vals, bins=bins)
    sum_sf, _ = np.histogram(x_vals, bins=bins, weights=sf_vals)
    sum_sf2, _ = np.histogram(x_vals, bins=bins, weights=sf_vals**2)
    
    # Safely divide to get mean and variance
    mean_sf = np.divide(sum_sf, counts, out=np.zeros_like(sum_sf), where=counts!=0)
    mean_sf2 = np.divide(sum_sf2, counts, out=np.zeros_like(sum_sf2), where=counts!=0)
    
    variance = mean_sf2 - (mean_sf**2)
    # Standard error of the mean = std_dev / sqrt(N)
    std_err = np.sqrt(np.maximum(variance, 0)) / np.sqrt(np.maximum(counts, 1))
    
    # 5. Plotting
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Mask out empty bins for plotting
    valid = counts > 0 
    
    ax.errorbar(bin_centers[valid], mean_sf[valid], 
                xerr=x_err[valid], yerr=std_err[valid], 
                fmt='ko', markersize=5, label='MC Average SF')
    
    # Styling
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.7)
    ax.set_xlabel(x_var)
    ax.set_ylabel(f"Average {sf_var}")
    
    # Dynamically set y-limits to zoom in on the SF variations
    y_mean = np.mean(mean_sf[valid])
    ax.set_ylim(y_mean * 0.85, y_mean * 1.15) 
    
    hep.cms.label("Preliminary", data=False, lumi=LUMI, ax=ax, com=13.6)
    ax.legend(loc='best')
    
    plt.savefig(output_filename, bbox_inches='tight')
    plt.close(fig)
    print(f" -> Saved {output_filename}")
    
# ==========================================
# 6. PLOT SCALE FACTORS VS KINEMATICS
# ==========================================
# Load just the nominal data (no systematics needed for base profile)
print("\n--- Extracting Nominal Data for SF Profiling ---")
nominal_data = load_and_cut_data(variation='nominal')
print(nominal_data['TTBar'].columns)
# Plot SF vs pT
plot_sf_profile(
    data_dict=nominal_data, 
    x_var='muon_pt', 
    sf_var='mu_trig_sf', 
    output_filename="MuTrigSF_vs_Pt.pdf"
)

# Plot SF vs Eta
plot_sf_profile(
    data_dict=nominal_data, 
    x_var='muon_eta', 
    sf_var='mu_trig_sf', 
    output_filename="MuTrigSF_vs_Eta.pdf"
)


--- Extracting Nominal Data for SF Profiling ---
Extracting 'nominal' from 5 matching files...
Index(['nPV', 'nPVGood', 'MET_pt', 'MET_phi', 'lep_pt', 'ele_pt', 'muon_pt',
       'lep_eta', 'ele_eta', 'muon_eta', 'n_ak4', 'n_bjet', 'n_ak8', 'jet1_pt',
       'jet2_pt', 'bjet1_pt', 'bjet2_pt', 'jet1_eta', 'jet2_eta', 'bjet1_eta',
       'bjet2_eta', 'jet1_btag', 'jet2_btag', 'bjet1_btag', 'bjet2_btag',
       'fatjet1_pt', 'fatjet1_eta', 'fatjet1_mass', 'n_b_outZH', 'n_ak4jets',
       'norm_weight', 'genWeight', 'topptWeight', 'topptWeight_Up',
       'topptWeight_Down', 'ele_reco_sf', 'ele_reco_sfup', 'ele_reco_sfdown',
       'ele_id_sf', 'ele_id_sfup', 'ele_id_sfdown', 'mu_id_sf', 'mu_id_sfup',
       'mu_id_sfdown', 'mu_iso_sf', 'mu_iso_sfup', 'mu_iso_sfdown',
       'mu_trig_sf', 'mu_trig_sfup', 'mu_trig_sfdown', 'puWeight',
       'puWeight_up', 'puWeight_down', 'isr_up', 'isr_down', 'fsr_up',
       'fsr_down', 'mu_r_up', 'mu_r_down', 'mu_f_up', 'mu_f_down', 'mu_rf_up',
       

In [16]:
def plot_sf_eta_pt_map(data_dict, sf_var='mu_trig_sf', output_filename="MuTrigSF_EtaPt_Map.pdf"):
    """
    Plots a 2D Map (Eta vs pT) where the color represents the average Scale Factor.
    """
    print(f"Creating 2D Eta-pT map for {sf_var}...")
    
    # 1. Combine all MC processes into a single DataFrame
    mc_dfs = []
    for proc in bkg_processes + sig_processes:
        if data_dict.get(proc) is not None and not data_dict[proc].empty:
            mc_dfs.append(data_dict[proc])
            
    if not mc_dfs:
        print("No MC data found to plot!")
        return
        
    df_mc = pd.concat(mc_dfs, ignore_index=True)
    
    # --- Data Validations ---
    if sf_var not in df_mc.columns:
        print(f"🚨 WARNING: '{sf_var}' not found in the DataFrame! Skipping 2D plot.")
        return
        
    if 'muon_pt' not in df_mc.columns or 'muon_eta' not in df_mc.columns:
        print(f"🚨 WARNING: Kinematic variables missing! Check validation_vars.")
        return

    # 2. Extract and clean values
    x_var, y_var = 'muon_eta', 'muon_pt'
    mask = df_mc[x_var].notna() & df_mc[y_var].notna() & df_mc[sf_var].notna()
    
    x_vals = df_mc[x_var][mask].values
    y_vals = df_mc[y_var][mask].values
    sf_vals = df_mc[sf_var][mask].values
    
    # 3. Get binning 
    # Example: 12 bins for eta (width of 0.4 per bin)
    nx, xmin, xmax = (12, -2.4, 2.4) 
    
    # Example: 10 bins for pT (width of 50 GeV per bin, capped at 500 to avoid empty high-pT bins)
    ny, ymin, ymax = (16, 0, 800)
    
    x_bins = np.linspace(xmin, xmax, nx + 1)
    y_bins = np.linspace(ymin, ymax, ny + 1)
    
    # 4. Calculate 2D Means using numpy
    counts, x_edges, y_edges = np.histogram2d(x_vals, y_vals, bins=[x_bins, y_bins])
    sum_sf, _, _ = np.histogram2d(x_vals, y_vals, bins=[x_bins, y_bins], weights=sf_vals)
    
    # Divide sum by counts to get the mean, avoiding division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        mean_sf = np.where(counts > 0, sum_sf / counts, np.nan)
        
    # 5. Plotting
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Transpose mean_sf (.T) because pcolormesh expects (Y, X)
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad(color='white') # Set bins with 0 stats to white
    
    im = ax.pcolormesh(x_edges, y_edges, mean_sf.T, cmap=cmap, shading='flat')
    
    # Add Colorbar
    cbar = fig.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label(f"Average {sf_var}", fontsize=18)
    
    # Styling
    ax.set_xlabel(r"Muon $\eta$")
    ax.set_ylabel(r"Muon $p_{T}$ [GeV]")
    
    # Optional: Set a logical max limit for pT if you have a long tail
    # ax.set_ylim(ymin, 500) 
    
    hep.cms.label("Preliminary", data=False, lumi=LUMI, ax=ax, com=13.6)
    
    plt.savefig(output_filename, bbox_inches='tight')
    plt.close(fig)
    print(f" -> Saved {output_filename}")
plot_sf_eta_pt_map(
    data_dict=nominal_data, 
    sf_var='mu_trig_sf', 
    output_filename="MuTrigSF_EtaPt_2D_Map.pdf"
)

Creating 2D Eta-pT map for mu_trig_sf...
 -> Saved MuTrigSF_EtaPt_2D_Map.pdf
